# Aula 6 — Alocação: Markowitz e a Fronteira Eficiente

**Intensivão Quant AI — ImpactUFSCar**

---

Até a Aula 5, usamos a alocação **Equal-Weight (1/N)**: distribuímos pesos iguais (10%) para as Top 10 ações selecionadas pelo sinal. Embora seja um benchmark extremamente robusto, hoje vamos explorar o clássico modelo quantitativo de otimização de portfólios: a **Teoria Moderna de Portfólio (Markowitz 1952)**.

Ao final desta aula, você terá aprendido:
- O portfólio sob a ótica da álgebra linear: retorno esperado $\mathbf{w}^\top \boldsymbol{\mu}$ e variância $\mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$.
- Por que a otimização de **Markowitz Pura** é instável, concentrada e impraticável devido a erros de estimação amostral.
- Como a **Otimização de Markowitz Restrita** (impondo limites rígidos de alocação de 0% a 20%) resolve a instabilidade e atua como uma solução robusta na prática.
- Como implementar otimização não-linear usando o solver `scipy.optimize.minimize`.
- Como avaliar o **Turnover** (rotação de carteira) e seu impacto direto em custos operacionais.

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.
# Qualquer pessoa que clonar o repositório pode rodar sem modificações.

import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow',
                    'statsmodels', 'python-docx', 'anthropic'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ], check=False)
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Garante que pyarrow está instalado (necessário para ler/salvar parquet)
    try:
        import pyarrow
    except ImportError:
        print("Instalando pyarrow...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'],
                       check=False)

    # Localiza a raiz do repositório subindo a árvore de diretórios até o .git
    # Funciona independente de onde o usuário clonou o projeto
    _p = os.path.abspath(os.getcwd())
    _root = None
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            _root = _p
            break
        _p = os.path.dirname(_p)

    # Fallback: se não encontrar .git, usa a pasta pai do notebook
    if _root is None:
        _root = os.path.dirname(os.path.abspath('__file__'))
        print("  Aviso: repositório .git não encontrado. Usando pasta do notebook.")

    DADOS_DIR = os.path.join(_root, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados em: {DADOS_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Carregar dados gerados nas aulas anteriores
ret_diarios = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_diarios_limpo.parquet'))
ret_mensais = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
sinal_v1    = pd.read_parquet(os.path.join(DADOS_DIR, 'sinal_v1.parquet'))

# Equal Weight (gerado na Aula 5)
pesos_ew = pd.read_parquet(os.path.join(DADOS_DIR, 'pesos_v1.parquet'))

# Baixar o IBOVESPA como Benchmark de mercado
ibov_raw = yf.download('^BVSP', start='2012-01-01', end='2024-12-31', auto_adjust=True, progress=False)['Close'].squeeze()
ret_ibov = np.log(ibov_raw / ibov_raw.shift(1)).resample('ME').sum()
ret_ibov = ret_ibov.reindex(sinal_v1.index).fillna(0.0)

print("Dados carregados:")
print(f"  Retornos diários: {ret_diarios.shape}")
print(f"  Retornos mensais: {ret_mensais.shape}")
print(f"  Sinal v1:         {sinal_v1.shape}")

In [ ]:
# ── VERIFICAÇÃO DE DEPENDÊNCIAS ──────────────────────────────────────────────
# Este notebook depende dos dados gerados pela aula anterior.
# Se algum arquivo estiver faltando, rode o notebook indicado primeiro.

_arquivos_necessarios = [
    os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'),
    os.path.join(DADOS_DIR, 'retornos_diarios_limpo.parquet'),
    os.path.join(DADOS_DIR, 'sinal_v1.parquet'),
    os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet'),
]

_faltando = [f for f in _arquivos_necessarios if not os.path.exists(f)]
if _faltando:
    print("\n⚠  ATENÇÃO: arquivos necessários não encontrados:")
    for f in _faltando:
        print(f"   Faltando: {os.path.basename(f)}")
    print(f"\n   Execute primeiro o notebook: aula-05-backtest-v1")
    print(f"   Os dados devem ficar em: {DADOS_DIR}")
else:
    print("✓  Todos os arquivos necessários encontrados.")

## 1. O Portfólio como Álgebra Linear

Ao decidirmos como alocar nosso capital entre as 10 ações selecionadas pela nossa estratégia de momentum, passamos da simples contabilidade para o domínio da **álgebra linear**. Em vez de pensarmos em pesos de forma isolada, nós modelamos o portfólio como vetores e matrizes para calcular com precisão matemática o retorno esperado e o risco total da carteira.

---

### A Matemática de Vetores e Matrizes do Portfólio

Para um portfólio contendo $N$ ativos selecionados:

1. **Vetor de Pesos ($w$):** Uma matriz coluna contendo o percentual do patrimônio líquido alocado em cada ativo:
   $$w = \begin{bmatrix} w_1 \\ w_2 \\ \vdots \\ w_N \end{bmatrix}$$
   Com a restrição física de que não podemos usar mais dinheiro do que possuímos (portfólio totalmente investido):
   $$\sum_{i=1}^N w_i = w^T \mathbf{1} = 1$$

2. **Vetor de Retornos Esperados ($\mu$):** Contendo as estimativas de retornos futuros de cada ativo:
   $$\mu = \begin{bmatrix} \mu_1 \\ \mu_2 \\ \vdots \\ \mu_N \end{bmatrix}$$

3. **Matriz de Covariância ($\Sigma$):** Uma matriz simétrica $N \times N$ que descreve a variabilidade individual de cada ativo (na diagonal principal) e a co-relação de comportamento estatístico entre todos os pares de ativos (fora da diagonal):
   $$\Sigma = \begin{bmatrix}
   \sigma_1^2 & \sigma_{1,2} & \dots & \sigma_{1,N} \\
   \sigma_{2,1} & \sigma_2^2 & \dots & \sigma_{2,N} \\
   \vdots & \vdots & \ddots & \vdots \\
   \sigma_{N,1} & \sigma_{N,2} & \dots & \sigma_N^2
   \end{bmatrix}$$

---

### O Retorno e Risco Resultantes da Carteira

Usando multiplicações de matrizes, a modelagem se torna extremamente compacta:

#### Retorno Esperado da Carteira ($\mu_p$):
$$\mu_p = w^T \mu = \sum_{i=1}^N w_i \mu_i$$

#### Variância da Carteira ($\sigma_p^2$):
$$\sigma_p^2 = w^T \Sigma w = \sum_{i=1}^N \sum_{j=1}^N w_i w_j \sigma_{i,j}$$

**Por que isso importa para iniciantes?**
O risco total da carteira ($\sigma_p^2$) não é simplesmente a média ponderada da volatilidade individual dos ativos! Graças à **diversificação**, se os ativos selecionados tiverem correlações baixas entre si (ou seja, $\sigma_{i,j}$ pequenos ou negativos), a volatilidade final da carteira será significativamente **menor** do que a volatilidade individual média dos ativos que a compõem. Este é o único "almoço grátis" no mercado financeiro, descoberto por Harry Markowitz.


In [ ]:
data_exemplo = '2022-12-30'
s = sinal_v1.loc[data_exemplo].dropna()
top_10 = s.nlargest(10).index.tolist()

# Histórico diário recente para calcular a covariância ex-ante
janela_cov = ret_diarios.loc[:data_exemplo].iloc[-252:][top_10].dropna(axis=1)
ativos = janela_cov.columns.tolist()
n = len(ativos)
Sigma = janela_cov.cov().values * 252  # Anualizada

# Diferentes distribuições de pesos
w_ew = np.ones(n) / n
w_concentrado = np.zeros(n); w_concentrado[0] = 1.0

def var_portfolio(w, Sigma):
    return w @ Sigma @ w

print(f"Data de análise: {data_exemplo}")
print(f"Variância ex-ante Anualizada:")
print(f"  Equal Weight (1/N):    {var_portfolio(w_ew, Sigma):.4f} -> Vol = {np.sqrt(var_portfolio(w_ew, Sigma)):.2%}")
print(f"  100% no Ativo {ativos[0]}: {var_portfolio(w_concentrado, Sigma):.4f} -> Vol = {np.sqrt(var_portfolio(w_concentrado, Sigma)):.2%}")

## 2. A Otimização de Markowitz Pura (Máximo Sharpe)

Em 1952, Harry Markowitz publicou seu artigo revolucionário que deu origem à **Moderna Teoria do Portfólio (MPT)**. O conceito principal é encontrar a combinação de pesos dos ativos que otimiza a relação de risco e retorno.

---

### A Formulação Matemática do Problema

O objetivo clássico é encontrar a carteira que possui o **Máximo Sharpe Ratio** no histórico. Isso equivale a maximizar a inclinação da reta que liga a taxa livre de risco à carteira na fronteira eficiente.

Formalmente, o problema de otimização é escrito como:

$$\max_{w} \quad \text{Sharpe}(w) = \frac{w^T \mu - R_f}{\sqrt{w^T \Sigma w}}$$

Sujeito às restrições fundamentais:
1. **Capital totalmente investido:** $\sum_{i=1}^N w_i = 1$
2. **Long-Only (Apenas Comprado):** $w_i \ge 0 \quad \forall i$ (não é permitido vender ativos a descoberto nesta formulação padrão).

### Como resolvemos no Python?
Como o Sharpe Ratio é uma função não linear de $w$, não existe uma solução analítica simples de álgebra de caneta e papel para o caso restrito long-only. Por isso, usamos o solucionador numérico robusto `scipy.optimize.minimize` para buscar o vetor de pesos $w$ ideal que minimiza o negativo do Sharpe Ratio.


In [ ]:
def pesos_markowitz_puro(sinal, ret_mensais, n_top=10, janela_hist=36):
    """
    Otimização clássica de Markowitz Long-Only (pesos de 0 a 1) para as Top N ações.
    Estimamos mu e Sigma com base na janela histórica de retornos mensais.
    """
    pesos = pd.DataFrame(0.0, index=sinal.index, columns=sinal.columns)
    
    for i, data in enumerate(sinal.index):
        if i < janela_hist: 
            continue
        
        # Selecionar Top 10 ativos conforme o sinal
        s = sinal.loc[data].dropna()
        if len(s) < n_top: 
            continue
        top = s.nlargest(n_top).index
        
        # Retornos históricos in-sample da janela móvel
        hist = ret_mensais.loc[:data].tail(janela_hist)[top].dropna(axis=1)
        if hist.shape[1] < 3: 
            continue
            
        mu = hist.mean().values
        cov = hist.cov().values
        n = len(mu)
        
        # Função objetivo: minimizar Sharpe negativo (equivale a maximizar Sharpe)
        def neg_sharpe(w):
            ret = w @ mu
            vol = np.sqrt(w @ cov @ w)
            return -ret / vol if vol > 1e-8 else 0.0
            
        # Restrições
        constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
        bounds = [(0.0, 1.0)] * n  # Long-only sem limite individual extra (0% a 100%)
        w0 = np.ones(n) / n        # Chute inicial uniforme
        
        res = optimize.minimize(neg_sharpe, w0, method='SLSQP', bounds=bounds, constraints=constraints)
        
        if res.success:
            pesos.loc[data, hist.columns] = res.x
            
    return pesos

print("Computando pesos para Markowitz Puro...")
pw_puro = pesos_markowitz_puro(sinal_v1, ret_mensais, n_top=10, janela_hist=36)
print("Concluído!")

## 3. Markowitz Restrito (A Solução Industrial contra a Instabilidade)

Embora a Otimização de Markowitz Pura seja matematicamente brilhante e elegante, **sua aplicação direta sem limites individuais na vida real costuma ser um desastre completo!** Vamos entender por que isso acontece e como resolvemos o problema na indústria.

---

### O Grande Problema de Markowitz: O Erro de Estimação de Parâmetros

A otimização de Markowitz é conhecida nos círculos de quants profissionais como um **"amplificador de erros de estimativa"**.
- O otimizador numérico assume que os valores históricos de média ($\mu$) e matriz de covariância ($\Sigma$) calculados no passado serão **exatamente idênticos** no futuro.
- No mundo real, estimar o retorno esperado futuro das ações é uma tarefa cheia de ruído estatístico. A média histórica é uma estimadora péssima e altamente instável dos retornos futuros.
- **O resultado da Otimização Pura:** O algoritmo numérico aloca pesos absurdamente extremos (por exemplo, aloca $95\%$ da carteira em um único ativo que teve uma alta histórica temporária fantástica e coloca quase $0\%$ nos outros).
- **Consequência:** A carteira perde toda a diversificação, torna-se extremamente concentrada e exibe um **turnover explosivo** (no mês seguinte, o peso de $95\%$ é destruído e recriado em outra ação, gerando custos de corretagem abusivos que devoram toda a rentabilidade).

---

### O Markowitz Restrito como Regularizador Matemático (Ridge/L2)

Para contornar a fragilidade clássica da estimativa de parâmetros, os quants institucionais impõem **limites superiores rígidos (bounds)** à alocação de cada ativo individual. 
Nesta aula, aplicamos a restrição:

$$0.0 \le w_i \le 0.20 \quad \forall i$$

Isso significa que **nenhum ativo individual pode compor mais do que $20\%$ do total da carteira**.

#### Por que isso funciona maravilhosamente?
1. **Garantia de Diversificação:** Forçamos o portfólio a se espalhar por pelo menos 5 ativos diferentes, mesmo que um deles pareça historicamente "perfeito".
2. **Estabilização out-of-sample:** A imposição dessas restrições de limite superior atua de forma análoga a uma **regularização Ridge ou L2** no aprendizado de máquina. Ela restringe o espaço de busca dos pesos, diminuindo drasticamente a variância out-of-sample dos retornos do portfólio.
3. **Controle de Custos operacionais:** Ao reduzir os pesos extremos, o giro da carteira (turnover) cai drasticamente de um período para o outro, reduzindo o impacto de mercado e as taxas de corretagem no rebalanceamento real.


In [ ]:
def pesos_markowitz_restrito(sinal, ret_mensais, n_top=10, janela_hist=36):
    """
    Otimização de Markowitz com restrição rígida de peso por ativo (0% a 20%).
    Isso mitiga a extrema sensibilidade aos parâmetros amostrais.
    """
    pesos = pd.DataFrame(0.0, index=sinal.index, columns=sinal.columns)
    
    for i, data in enumerate(sinal.index):
        if i < janela_hist: 
            continue
        
        # Selecionar Top 10 ativos conforme o sinal
        s = sinal.loc[data].dropna()
        if len(s) < n_top: 
            continue
        top = s.nlargest(n_top).index
        
        # Retornos históricos in-sample da janela móvel
        hist = ret_mensais.loc[:data].tail(janela_hist)[top].dropna(axis=1)
        if hist.shape[1] < 3: 
            continue
            
        mu = hist.mean().values
        cov = hist.cov().values
        n = len(mu)
        
        # Função objetivo: minimizar Sharpe negativo
        def neg_sharpe(w):
            ret = w @ mu
            vol = np.sqrt(w @ cov @ w)
            return -ret / vol if vol > 1e-8 else 0.0
            
        # Restrições
        constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
        bounds = [(0.0, 0.20)] * n  # LIMITAÇÃO CRUCIAL: Máximo 20% por ativo!
        w0 = np.ones(n) / n
        
        res = optimize.minimize(neg_sharpe, w0, method='SLSQP', bounds=bounds, constraints=constraints)
        
        if res.success:
            pesos.loc[data, hist.columns] = res.x
            
    return pesos

print("Computando pesos para Markowitz Restrito...")
pw_restrito = pesos_markowitz_restrito(sinal_v1, ret_mensais, n_top=10, janela_hist=36)
print("Concluído!")

## 4. Comparação de Performance e Curvas de Retorno

Para validar se as otimizações trouxeram benefício prático, vamos simular o retorno mensal do portfólio aplicando os pesos de cada estratégia com o clássico atraso de rebalanceamento `shift(1)` para evitar viés de antecipação (look-ahead bias).

In [ ]:
def retorno_carteira(pesos_df, ret_mensais):
    # Shift(1) aplica os pesos do final do mês T no retorno do mês T+1
    ret = (pesos_df.shift(1) * ret_mensais).sum(axis=1)
    return ret[pesos_df.sum(axis=1) > 0]

ret_ew = retorno_carteira(pesos_ew, ret_mensais)
ret_puro = retorno_carteira(pw_puro, ret_mensais)
ret_restrito = retorno_carteira(pw_restrito, ret_mensais)

# Sincronizar índices
idx = ret_ew.index.intersection(ret_puro.index).intersection(ret_restrito.index)
ret_ew, ret_puro, ret_restrito = ret_ew[idx], ret_puro[idx], ret_restrito[idx]
ret_bm = ret_ibov.reindex(idx).fillna(0.0)

# Funções auxiliares para métricas baseadas na Aula 5
from scipy.stats import linregress

def calcular_metricas(retornos, benchmark=None, nome='Estratégia'):
    n_anos    = len(retornos) / 12
    ret_total = np.exp(retornos.sum()) - 1
    cagr      = (1 + ret_total) ** (1 / n_anos) - 1
    vol_anual = retornos.std() * np.sqrt(12)
    sharpe    = (retornos.mean() / retornos.std()) * np.sqrt(12)
    valor     = np.exp(retornos.cumsum())
    mdd       = ((valor - valor.cummax()) / valor.cummax()).min()
    return dict(nome=nome, cagr=cagr, vol_anual=vol_anual, sharpe=sharpe, max_drawdown=mdd)

def exibir_metricas(*ms):
    fmt = dict(cagr='{:.1%}', vol_anual='{:.1%}', sharpe='{:.2f}', max_drawdown='{:.1%}')
    labels = dict(cagr='CAGR', vol_anual='Vol Anual', sharpe='Sharpe', max_drawdown='Max Drawdown')
    nomes = [d['nome'] for d in ms]
    w = max(14, max(len(n) for n in nomes) + 2)
    h = f"{'Métrica':<22}" + ''.join(f"{n:>{w}}" for n in nomes)
    print(h); print('─' * len(h))
    for k, lb in labels.items():
        linha = f"{lb:<22}"
        for d in ms:
            v = d.get(k, np.nan)
            s = fmt[k].format(v) if not (isinstance(v, float) and np.isnan(v)) else 'N/A'
            linha += f"{s:>{w}}"
        print(linha)

m_ew = calcular_metricas(ret_ew, ret_bm, 'Equal Weight')
m_puro = calcular_metricas(ret_puro, ret_bm, 'Markowitz Puro')
m_rest = calcular_metricas(ret_restrito, ret_bm, 'Markowitz Restrito')
m_ibov = calcular_metricas(ret_bm, nome='IBOVESPA')

exibir_metricas(m_ew, m_puro, m_rest, m_ibov)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
np.exp(ret_ew.cumsum()).plot(ax=ax, label='Equal Weight', linewidth=2)
np.exp(ret_puro.cumsum()).plot(ax=ax, label='Markowitz Puro', linewidth=2)
np.exp(ret_restrito.cumsum()).plot(ax=ax, label='Markowitz Restrito', linewidth=2, color='darkgoldenrod')
np.exp(ret_bm.cumsum()).plot(ax=ax, label='IBOVESPA', linewidth=1.5, linestyle='--', color='gray')

ax.set_title('Evolução do Capital Acumulado (Bruto) — Comparação de Alocação')
ax.set_ylabel('Patrimônio Acumulado (Base 1.0)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Análise de Rotação de Carteira (Turnover)

Métricas de performance bruta são enganosas. No mundo real, rebalancear carteiras incorre em custos operacionais e corretagem. O **Turnover** mensal mede a porcentagem de capital que precisa ser rotacionada de um mês para outro:
$$\text{Turnover}_t = \frac{\sum_{i=1}^N |w_{i, t} - w_{i, t-1}|}{2}$$

Um portfólio que muda excessivamente de pesos de forma brusca gerará custos operacionais muito altos, destruindo os retornos líquidos.

In [ ]:
def calcular_turnover(pesos_df):
    # Soma absoluta da diferença de pesos / 2
    delta = pesos_df.diff().abs().sum(axis=1) / 2.0
    return delta[delta > 0.0]

to_ew = calcular_turnover(pesos_ew)
to_puro = calcular_turnover(pw_puro)
to_rest = calcular_turnover(pw_restrito)

print('Turnover Mensal Médio:')
print(f'  Equal-Weight (1/N):    {to_ew.mean():.1%}')
print(f'  Markowitz Puro:        {to_puro.mean():.1%}')
print(f'  Markowitz Restrito:    {to_rest.mean():.1%}')

# Plot comparativo do turnover móvel
fig, ax = plt.subplots(figsize=(13, 4))
to_ew.plot(ax=ax, label='Equal Weight', alpha=0.7)
to_puro.plot(ax=ax, label='Markowitz Puro', alpha=0.7)
to_rest.plot(ax=ax, label='Markowitz Restrito', alpha=0.7, color='darkgoldenrod')

ax.set_title('Evolução Mensal do Turnover de Rebalanceamento')
ax.set_ylabel('Rotação Operacional (%)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

## 6. Salvando Resultados para o Pipeline

Como a otimização de **Markowitz Restrita** ofereceu o melhor equilíbrio entre robustez amostral (Sharp Ratio elevado out-of-sample) e estabilidade de pesos (reduzido turnover operacional comparado ao Puro), vamos selecioná-lo como a nossa carteira para a próxima aula.

Salvaremos os pesos e retornos da carteira nas variáveis globais do pipeline como `pesos_v2.parquet` e `retorno_carteira_v2.parquet`.

In [ ]:
# Salvar os pesos e retornos calculados
pw_restrito.to_parquet(os.path.join(DADOS_DIR, 'pesos_v2.parquet'))
ret_restrito.to_frame('retorno').to_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_v2.parquet'))

print("Arquivos salvos com sucesso no diretório de dados:")
print(f"  -> pesos_v2.parquet ({os.path.join(DADOS_DIR, 'pesos_v2.parquet')})")
print(f"  -> retorno_carteira_v2.parquet ({os.path.join(DADOS_DIR, 'retorno_carteira_v2.parquet')})")

## Resumo da Aula

| Conceito | Implementação / Fórmula |
|---|---|
| **Retorno do Portfólio** | $r_p = \mathbf{w}^\top \mathbf{r}$ |
| **Variância do Portfólio** | $\sigma_p^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$ |
| **Markowitz Puro** | $\max \text{Sharpe}$ com $w_i \in [0, 1]$ |
| **Markowitz Restrito** | $\max \text{Sharpe}$ com $w_i \in [0, 0.20]$ (Solução Industrial) |
| **Turnover** | $\sum |w_{i, t} - w_{i, t-1}| / 2$ |

---

## Para Praticar em Casa

1. **Teste a Sensibilidade da Janela**: Altere `janela_hist=36` para `janela_hist=12` e para `janela_hist=60` na otimização de Markowitz Restrita. Veja o que acontece com o Sharpe e o Turnover.
2. **Modifique o Cap**: Mude a restrição de peso máximo de 20% para 10% (bounds de 0 a 10%) e depois para 50%. Avalie se portfólios mais concentrados de fato melhoram a performance out-of-sample ou apenas geram mais risco.

---

*ImpactUFSCar — Diretoria de Quant*